# Lily Wan Kaggle — Clean v2\n\nUse **GPU T4 ×2**. Run Cell 1, wait for `INSTALL DONE ✨`, then run Cell 2. Cell 2 does everything else and prints the `gradio.live` link.\n

In [ ]:
# CELL 1 — install only
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg safetensors

print("INSTALL DONE ✨")


In [ ]:
# CELL 2 — EVERYTHING ELSE
# Load Wan + define generator + launch Gradio in ONE cell.
# Do not split this cell.

import os, gc, uuid, traceback
import torch
import gradio as gr
from PIL import Image
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"

if not torch.cuda.is_available():
    raise RuntimeError("GPU is OFF. Kaggle → Settings → Accelerator → GPU T4 x2.")

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPUs:", gpu_names)

print("\nLoading Wan. First launch can take a while; later launches use Kaggle's cache.")

try:
    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="balanced",
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    print("Model loaded with balanced device mapping.")
except Exception as e:
    print("Balanced loading failed; falling back to CPU offload.")
    print("Reason:", repr(e))
    gc.collect()
    torch.cuda.empty_cache()

    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    pipe.enable_model_cpu_offload()

try:
    pipe.enable_vae_tiling()
except Exception:
    pass

try:
    pipe.enable_vae_slicing()
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()
print("WAN READY ✨")

def fit_image(img):
    img = img.convert("RGB")
    if img.height >= img.width:
        width, height = 256, 448
    else:
        width, height = 448, 256

    source_ratio = img.width / img.height
    target_ratio = width / height

    if source_ratio > target_ratio:
        crop_w = int(img.height * target_ratio)
        left = (img.width - crop_w) // 2
        img = img.crop((left, 0, left + crop_w, img.height))
    else:
        crop_h = int(img.width / target_ratio)
        top = (img.height - crop_h) // 2
        img = img.crop((0, top, img.width, top + crop_h))

    return img.resize((width, height), Image.LANCZOS), width, height

def generate_video(image, prompt, seconds, steps, seed):
    try:
        if image is None:
            return None, "Upload an image first."

        if not prompt or not prompt.strip():
            return None, "Type a motion prompt first."

        image, width, height = fit_image(image)
        num_frames = 17 if int(seconds) == 2 else 25

        seed = int(seed)
        if seed <= 0:
            seed = int.from_bytes(os.urandom(4), "little")

        generator = torch.Generator(device="cpu").manual_seed(seed)

        gc.collect()
        torch.cuda.empty_cache()

        print(
            f"\nGenerating: {width}x{height}, "
            f"{num_frames} frames, {int(steps)} steps, seed {seed}"
        )

        result = pipe(
            prompt=prompt.strip(),
            reference_images=image,
            height=height,
            width=width,
            num_frames=num_frames,
            num_inference_steps=int(steps),
            guidance_scale=5.0,
            generator=generator,
        ).frames[0]

        out_dir = "/kaggle/working/lily_videos"
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"wan_{uuid.uuid4().hex[:8]}.mp4")

        export_to_video(result, path, fps=8)

        gc.collect()
        torch.cuda.empty_cache()

        return path, f"Finished ✨  seed: {seed}"

    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        msg = (
            "CUDA ran out of memory. Try 2 seconds and 6 steps. "
            "If it keeps happening, restart the Kaggle session and run these two cells again."
        )
        print(msg)
        return None, msg

    except Exception as e:
        traceback.print_exc()
        msg = f"{type(e).__name__}: {e}"
        return None, msg

with gr.Blocks() as app:
    gr.Markdown("# ✦ Lily Video Studio")
    gr.Markdown(
        "Wan is running on your Kaggle GPU. "
        "Start with **2 seconds / 6–8 steps**."
    )

    image_input = gr.Image(type="pil", label="Input image")

    prompt_input = gr.Textbox(
        lines=4,
        label="Motion prompt",
        placeholder=(
            "natural full-body movement, smooth motion, realistic hair and fabric movement, "
            "stable face, coherent anatomy"
        ),
    )

    with gr.Row():
        subtle = gr.Button("Subtle")
        cinematic = gr.Button("Cinematic")
        dance = gr.Button("Dance")

    subtle.click(
        lambda: (
            "subtle natural breathing, blinking, gentle head movement, "
            "tiny realistic body movement, stable camera, coherent anatomy"
        ),
        outputs=prompt_input,
    )

    cinematic.click(
        lambda: (
            "slow cinematic camera movement, natural body motion, "
            "soft hair and fabric movement, realistic coherent motion"
        ),
        outputs=prompt_input,
    )

    dance.click(
        lambda: (
            "energetic full-body dancing, expressive natural movement, "
            "smooth coherent anatomy, dynamic but stable camera"
        ),
        outputs=prompt_input,
    )

    with gr.Accordion("Settings", open=False):
        seconds = gr.Radio([2, 3], value=2, label="Seconds")
        steps = gr.Slider(6, 12, value=8, step=1, label="Steps")
        seed = gr.Number(value=0, precision=0, label="Seed (0 = random)")

    generate_button = gr.Button("Generate video ✦", variant="primary")
    video_output = gr.Video(label="Result")
    status_output = gr.Textbox(label="Status", interactive=False)

    generate_button.click(
        fn=generate_video,
        inputs=[image_input, prompt_input, seconds, steps, seed],
        outputs=[video_output, status_output],
    )

print("\nUI READY ✨")
print("Opening Gradio share link now...")

app.queue(max_size=2).launch(share=True)
